# Ejercicio Tema 1: Análisis exploratorio de datos y visualización con matplotlib/seaborn

### Juan Alcaraz, José Aguilar, Óscar Camacho, Adrián Carrasco, Javier Herrero, Clara Montalvá 

## 1. Preprocesado y estadística descriptiva

In [ ]:
# Dataset : Arritmias.csv

import pandas as pd
import numpy as np

df_raw = pd.read_csv("data/Arritmias.csv")
cols = df_raw.columns[1:-4]

# Preproceso: 
#  a) cambiar separador decimal (, por .)
#  b) re-etiquetar AV como clase binaria (0/1)

# Cambio , por . en las series de excel formated_df: fdf
df = df_raw.copy()

for i in range(len(cols)):
      df[cols[i]] = df[cols[i]].str.replace(",", ".").astype(float)

cols = df.columns[1:-1]
X = df[cols].to_numpy()  # Marcadores de arritmia post-infarto
Y = df['AV'].to_numpy()  # AV: Arritmia Ventricular

print(df.head())

# Estudiar los 2 grupos de pacientes suministrados:  AV = 0 (no presenta arritmia) y AV = 1 (presenta arritmia)  
# Analizar y representar gráficamente la influencia de los marcadores suministrados en ambos grupos.
# Escribir un pequeño informe con el resumen de las gráficas generadas y las principales conclusiones extraídas (2-3 pp).

# Descripción de los marcadores utilizados :

# LV Mass(g)                : masa en gramos del ventrículo izquierdo.
# BZ+Core (g)               : masa en gramos de la zona infartada (core) + su borde (BZone)
# BZ (g, %)                 : masa en gramos de la zona de borde y su porcentaje
# Core (g, %)               : masa en gramos de la zona infartada y su porcentaje
# Channel_Mass (g)          : masa en gramos de los canales encontrados
# LVEF                      : fuerza de eyección del ventrículo
# Edad, Sexo
# AV                        : presenta arrimia (1), no presenta arritmia (0)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, ttest_ind
import warnings
warnings.filterwarnings('ignore')

# ── Paleta y estilo globales ──────────────────────────────────────────────────
COLOR_AV0  = '#4C72B0'   # azul  → sin arritmia
COLOR_AV1  = '#DD8452'   # naranja → con arritmia
PALETTE    = {0: COLOR_AV0, 1: COLOR_AV1}
LABEL_AV0  = 'AV=0 (sin arritmia)'
LABEL_AV1  = 'AV=1 (con arritmia)'

sns.set_theme(style='whitegrid', font_scale=1.05)

In [ ]:
# ── 1.1 Lectura del CSV ───────────────────────────────────────────────────────
df_raw = pd.read_csv('data/Arritmias.csv')

# ── 1.2 Identificación de columnas ───────────────────────────────────────────
# Columnas numéricas con coma decimal (todas salvo PACIENTES, SEXO, EDAD, AV)
comma_cols = ['LV MASS (g)', 'BZ + CORE (g)', 'BZ + CORE (%)',
              'BZ (g)', 'BZ (%)', 'CORE (g)', 'CORE (%)', 'CHANNEL MASS (g)']

# Variables demográficas y target
demo_cols   = ['EDAD', 'SEXO']
target_col  = 'AV'

# Marcadores cardíacos (los que se analizarán con Mann-Whitney)
marker_cols = comma_cols + ['LVEF']

# ── 1.3 Conversión a float (reemplazar coma decimal por punto) ────────────────
df = df_raw.copy()
for col in comma_cols:
    df[col] = df[col].astype(str).str.replace(',', '.').astype(float)

# Asegurar tipos numéricos en el resto
df['LVEF'] = pd.to_numeric(df['LVEF'], errors='coerce')
df['EDAD'] = pd.to_numeric(df['EDAD'], errors='coerce')
df['SEXO'] = pd.to_numeric(df['SEXO'], errors='coerce')   # 1=Hombre, 2=Mujer
df['AV']   = pd.to_numeric(df['AV'],   errors='coerce').astype(int)

# Separar grupos
df0 = df[df['AV'] == 0]
df1 = df[df['AV'] == 1]

print(f'Total pacientes: {len(df)}  |  AV=0: {len(df0)}  |  AV=1: {len(df1)}')
print('\nPrimeras filas:')
df.head()

In [ ]:
# ── 1.4 Estadística descriptiva por grupo ─────────────────────────────────────
stats_rows = []
for col in marker_cols:
    for av, grupo in [(0, df0), (1, df1)]:
        s = grupo[col]
        stats_rows.append({
            'Marcador': col, 'Grupo': f'AV={av}',
            'N': len(s), 'Media': round(s.mean(), 3),
            'Desv.Típica': round(s.std(), 3),
            'Min': round(s.min(), 3), 'P25': round(s.quantile(.25), 3),
            'Mediana': round(s.median(), 3), 'P75': round(s.quantile(.75), 3),
            'Max': round(s.max(), 3)
        })

desc_df = pd.DataFrame(stats_rows).set_index(['Marcador', 'Grupo'])
print('=== Estadística Descriptiva por Grupo ===')
display(desc_df)

In [ ]:
# ── 1.5 Test Mann-Whitney U por marcador ──────────────────────────────────────
def significance_label(p):
    if p < 0.001: return '*'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return 'ns'

mw_results = []
pvalues = {}   # guardamos para anotaciones en figuras
for col in marker_cols:
    stat, p = mannwhitneyu(df0[col].dropna(), df1[col].dropna(),
                           alternative='two-sided')
    pvalues[col] = p
    mw_results.append({
        'Marcador': col, 'U-statistic': round(stat, 2),
        'p-value': round(p, 5), 'Significancia': significance_label(p)
    })

mw_df = pd.DataFrame(mw_results).sort_values('p-value').set_index('Marcador')
print('=== Test Mann-Whitney U – Diferencias entre AV=0 y AV=1 ===')
display(mw_df)

# Marcadores significativos
sig_markers = [c for c in marker_cols if pvalues[c] < 0.05]
print(f'\nMarcadores significativos (p<0.05): {sig_markers}')

# Top-5 marcadores (menor p-value) para Figura 3
top5_markers = list(mw_df.head(5).index)

## 2. Figura 1 – Distribuciones por marcador (Violin + Box + Strip)

## 3. Figura 2 – Heatmap de correlaciones por grupo

## 4. Figura 3 – Pairplot de marcadores más discriminantes

## 5. Figura 4 – Variables demográficas (Edad y Sexo) vs AV

## 6. Figura 5 – Radar chart (Perfil medio normalizado)

## 7. Figura 6 – Scatter + Contornos KDE

## 8. Informe escrito